In [8]:
from dotenv import load_dotenv
from typing_extensions import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from typing import Annotated


load_dotenv()

True

In [9]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [10]:
class State(TypedDict):
    messages : Annotated[list, add_messages] 

def chatbot(state : State) -> State:
    return {"messages" : [llm.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node('chatbot_node' , chatbot)

builder.add_edge(START , "chatbot_node")
builder.add_edge("chatbot_node", END)


graph = builder.compile()

In [11]:
message = {"role" : "user" , "content" : "who walked on the moon for first time? Give only name"}


response = graph.invoke({"messages" : [message]})


response["messages"]

[HumanMessage(content='who walked on the moon for first time? Give only name', additional_kwargs={}, response_metadata={}, id='822b187a-54b0-4014-a101-415631cf6534'),
 AIMessage(content='Neil Armstrong.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 47, 'total_tokens': 51, 'completion_time': 0.013088176, 'completion_tokens_details': None, 'prompt_time': 0.004025868, 'prompt_tokens_details': None, 'queue_time': 0.052151446, 'total_time': 0.017114044}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaf05-5f40-74f3-ad68-af92e37bc695-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 4, 'total_tokens': 51})]

In [12]:
state = None
while True:
    in_message = input('You: ')

    if in_message.lower() in {'quit' ,'exit'}:
        break 

    if state is None:
        
        state: State = {
            "messages" : [{'role' : 'user', "content" : in_message}]
        }
    else:
        state["messages"].append({"role":"user", "content" : in_message})

    state = graph.invoke(state)
    print("Bot: " , state["messages"][-1].content)



Bot:  The first person to walk on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first person to set foot on the moon.
